In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Task 1: Write your code here:
ppath = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(ppath)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
df['Target'].isna().sum()
numerical_cols = df.columns.drop("Target")
print(numerical_cols)
for num in numerical_cols:
  print(num)
  df[num].fillna(df[num].mode()[0])


check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):


  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
#no string or categorical clmn so no encode needed
df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Pick only the numerical columns, NOT the target
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Target")

scaler = StandardScaler()

# scale the `numerical_cols`
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Target")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df["Target"].astype(float)

In [ ]:
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

In [ ]:
sr_results = {'loss': [], 'acc': [], 'f1': []}


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
n_splits = 5  # K=5 Folds
model = CatBoostClassifier()
# 5-Fold , shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  # Fit the model on train data
  model.fit(X_train, y_train)

  # Use the model to predict the test data
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  losses = y_test-y_pred
  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)


In [ ]:
# Task 1: Write your code here:
importance_model= list(zip(X.columns, model.feature_importances_))
sorted_importance = sorted(importance_model, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('catboost importance Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:
print(features[0])

In [ ]:
# Task Bonus: Write your code here:
X_golden = df['P_2']
n_splits = 5  # K=5 Folds
model = CatBoostClassifier()
# 5-Fold , shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X_golden, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  # Fit the model on train data
  model.fit(X_train, y_train)

  # Use the model to predict the test data
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  losses = y_test-y_pred
  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)